# Filtering AlphaProtein Novo Demo Designs

This notebook loads the per-design `*_evaluation.json` records produced by `evaluate_design.py` (located in the `04_eval/` subdirectory of a campaign output directory) and applies benchmark filtering thresholds for each of the four example enzyme design problems:

- Kemp eliminase (`examples/kemp_eliminase/`)
- 4MU-Ac serine esterase (`examples/serine_esterase/`)
- DEHP esterase (`examples/dehp_esterase_denovo/` and `examples/dehp_esterase_partial_diffusion/`)
- Nitrene transferase (`examples/nitrene_transferase/`)

## Filtering Utilities

In [ ]:
import enum
import json
from etils import epath
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


class Comparator(enum.Enum):
  GREATER_EQUAL = '>='
  LESS_EQUAL = '<='
  WITHIN_RANGE = 'within_range'


_C = Comparator


def evaluate_threshold(series: pd.Series, comparator: Comparator, value):
  if comparator == _C.GREATER_EQUAL:
    return series >= value
  if comparator == _C.LESS_EQUAL:
    return series <= value
  low, high = value
  return series.between(low, high)


def get_per_metric_masks(df: pd.DataFrame, thresholds: dict) -> pd.DataFrame:
  return pd.DataFrame({
      metric: evaluate_threshold(df[metric], comparator, value)
      for metric, (comparator, value) in thresholds.items()
  })


def get_filter_mask(df: pd.DataFrame, thresholds: dict) -> pd.Series:
  return get_per_metric_masks(df, thresholds).all(axis=1)


def summarize_thresholds(df: pd.DataFrame, thresholds: dict) -> pd.DataFrame:
  masks = get_per_metric_masks(df, thresholds)
  cumulative = masks.cumprod(axis=1).astype(bool)
  return pd.DataFrame({
      'metric': list(thresholds.keys()),
      'comparator': [c.value for c, _ in thresholds.values()],
      'threshold': [v for _, v in thresholds.values()],
      'pass_count': masks.sum().values,
      'pass_rate': masks.mean().values,
      'cumulative_pass_count': cumulative.sum().values,
      'cumulative_pass_rate': cumulative.mean().values,
  })


def load_eval_results(eval_dir: str) -> pd.DataFrame:
  records = []
  for path in sorted(epath.Path(eval_dir).glob('*_evaluation.json')):
    data = json.loads(path.read_text())
    records.append({
        'sample_name': data['sample_name'],
        'spec_name': data['spec_name'],
        'designed_cif': data['designed_cif'],
        **data['metrics'],
    })
  return pd.DataFrame(records)


def plot_pass_rates(summary_df: pd.DataFrame) -> None:
  sns.barplot(data=summary_df, x='pass_rate', y='metric')
  plt.xlim(0, 1)
  plt.show()

KEMP_ELIMINASE_THRESHOLDS = {
    'sampled/ligand_ca_neighbors_min': (_C.LESS_EQUAL, 21.0),
    'sampled/ligand_ca_neighbors_mean': (_C.GREATER_EQUAL, 18.0),
    'sampled/cat_base_tip_ca_neighbors_mean': (_C.GREATER_EQUAL, 15.0),
    'complex/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/seed_0': (_C.LESS_EQUAL, 10.0),
    'complex/protein_ligand_ipde_mean/seed_0': (_C.LESS_EQUAL, 2.0),
    'monomer/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'monomer/motif_bb_aligned_allatom_rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'complex/tm_score/median': (_C.GREATER_EQUAL, 80.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/median': (_C.LESS_EQUAL, 10.0),
    'complex/protein_ligand_ipde_mean/median': (_C.LESS_EQUAL, 2.0),
    'monomer/tm_score/median': (_C.GREATER_EQUAL, 80.0),
    'monomer/motif_bb_aligned_allatom_rmsd/median': (_C.LESS_EQUAL, 1.5),
    'complex/acid_base/distance/mean': (_C.LESS_EQUAL, 3.7),
    'complex/oxyanion_oah/distance/mean': (_C.LESS_EQUAL, 3.7),
    'complex/acid_base/dihedral2/mean': (_C.WITHIN_RANGE, (150.0, 210.0)),
    'complex/oxyanion_oah/dihedral2/mean': (_C.WITHIN_RANGE, (150.0, 210.0)),
}

SERINE_ESTERASE_BASE_THRESHOLDS = {
    'monomer/rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'monomer/protein_plddt_mean/seed_0': (_C.GREATER_EQUAL, 75.0),
    'monomer/motif_allatom_rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'monomer/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'monomer/predicted_distance_error/seed_0': (_C.LESS_EQUAL, 2.0),
    'complex/interface_lddt/seed_0': (_C.GREATER_EQUAL, 50.0),
    'complex/ligand_plddt_mean/seed_0': (_C.GREATER_EQUAL, 80.0),
    'complex/fixed_res_bb_aligned_allatom_sc_rmsd/seed_0': (_C.LESS_EQUAL, 3.0),
    'complex/motif_allatom_rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'complex/chain_pair_pde_mean_sym/seed_0': (_C.LESS_EQUAL, 2.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/seed_0': (_C.LESS_EQUAL, 5.0),
    'complex/mean_pocket_bb_rmsd/seed_0': (_C.LESS_EQUAL, 2.0),
    'sequence/seq_len': (_C.GREATER_EQUAL, 1),
    'sequence/max_single_aatype_frac': (_C.LESS_EQUAL, 0.25),
    'sequence/max_consecutive_identical_aatypes': (_C.LESS_EQUAL, 5),
    'sequence/num_blocklisted_motifs': (_C.LESS_EQUAL, 0),
    'sequence/unknown_count': (_C.LESS_EQUAL, 0),
    'sampled/topology_string_len': (_C.GREATER_EQUAL, 3),
    'sampled/num_missing_motif_residues': (_C.LESS_EQUAL, 0),
    'complex/num_missing_atoms/seed_0': (_C.LESS_EQUAL, 0),
}

SERINE_ESTERASE_HBOND_THRESHOLDS = {
    'monomer/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'monomer/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'es/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'es/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_fwd_oah1_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_fwd_oah2_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_fwd_oah1_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_fwd_oah2_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_fwd_oah1_ester/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_fwd_oah2_ester/mean': (_C.GREATER_EQUAL, 0.01),
}

SERINE_ESTERASE_THRESHOLDS = (
    SERINE_ESTERASE_BASE_THRESHOLDS
    | SERINE_ESTERASE_HBOND_THRESHOLDS
    | {
        'complex/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 90.4),
        'es_ti1_min_cross_state_rmsd': (_C.LESS_EQUAL, 1.0),
        'es_ti1_cross_seed_ligand_coord_std': (_C.LESS_EQUAL, 4.0),
    }
)

DEHP_ESTERASE_THRESHOLDS = (
    SERINE_ESTERASE_BASE_THRESHOLDS
    | SERINE_ESTERASE_HBOND_THRESHOLDS
    | {
        'complex/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 85.0),
        'aei/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 88.0),
        'ti2/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 88.0),
    }
)

DEHP_ESTERASE_STRICT_THRESHOLDS = DEHP_ESTERASE_THRESHOLDS | {
    'complex/ligand_plddt_mean/mean': (_C.GREATER_EQUAL, 92.0),
    'aei/ligand_plddt_mean/mean': (_C.GREATER_EQUAL, 90.0),
    'ti2/ligand_plddt_mean/mean': (_C.GREATER_EQUAL, 92.0),
}

NITRENE_TRANSFERASE_THRESHOLDS = {
    'sampled/percent_ligand_bb_clashes': (_C.LESS_EQUAL, 5.0),
    'monomer/predicted_distance_error/seed_0': (_C.LESS_EQUAL, 2.0),
    'monomer/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'complex/chain_pair_pde_mean_sym/seed_0': (_C.LESS_EQUAL, 2.0),
    'complex/interface_lddt/seed_0': (_C.GREATER_EQUAL, 30.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/seed_0': (
        _C.LESS_EQUAL,
        2.5,
    ),
    'complex/mean_pocket_bb_rmsd/seed_0': (_C.LESS_EQUAL, 2.0),
    'sequence/max_consecutive_identical_aatypes': (_C.LESS_EQUAL, 5),
    'sequence/max_single_aatype_frac': (_C.LESS_EQUAL, 0.25),
    'sequence/num_blocklisted_motifs': (_C.LESS_EQUAL, 0),
    'sequence/unknown_count': (_C.LESS_EQUAL, 0),
    'complex/expanded_motif/motif_allatom_rmsd/seed_0': (
        _C.LESS_EQUAL,
        1.5,
    ),
    'complex/expanded_motif/motif_with_ligand_allatom_rmsd/seed_0': (
        _C.LESS_EQUAL,
        2.5,
    ),
    'sequence/seq_len': (_C.GREATER_EQUAL, 90),
    'complex/ligand_plddt_mean/seed_0': (_C.GREATER_EQUAL, 75.0),
    'heme_substrate/pocket_C9_min_atom_dist/seed_0': (
        _C.WITHIN_RANGE,
        (2.8, 3.8),
    ),
    'heme_substrate/pocket_C9_allatom_neighbors_mean/seed_0': (
        _C.WITHIN_RANGE,
        (38.0, 62.0),
    ),
    'heme_substrate/propionate_hbond_num_atoms/seed_0': (
        _C.WITHIN_RANGE,
        (3, 6),
    ),
    'heme_substrate/propionate_hbond_num_res/seed_0': (
        _C.WITHIN_RANGE,
        (3, 5),
    ),
    'heme_substrate/his_heme_dist/seed_0': (_C.LESS_EQUAL, 3.0),
    'heme_substrate/piperidine_propensity/min': (_C.LESS_EQUAL, 0.35),
    'n_self_rmsd/seed_0': (_C.LESS_EQUAL, 0.8),
}

In [ ]:
KEMP_ELIMINASE_THRESHOLDS = {
    'sampled/ligand_ca_neighbors_min': (_C.LESS_EQUAL, 21.0),
    'sampled/ligand_ca_neighbors_mean': (_C.GREATER_EQUAL, 18.0),
    'sampled/cat_base_tip_ca_neighbors_mean': (_C.GREATER_EQUAL, 15.0),
    'complex/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/seed_0': (_C.LESS_EQUAL, 10.0),
    'complex/protein_ligand_ipde_mean/seed_0': (_C.LESS_EQUAL, 2.0),
    'monomer/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'monomer/motif_bb_aligned_allatom_rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'complex/tm_score/median': (_C.GREATER_EQUAL, 80.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/median': (_C.LESS_EQUAL, 10.0),
    'complex/protein_ligand_ipde_mean/median': (_C.LESS_EQUAL, 2.0),
    'monomer/tm_score/median': (_C.GREATER_EQUAL, 80.0),
    'monomer/motif_bb_aligned_allatom_rmsd/median': (_C.LESS_EQUAL, 1.5),
    'complex/acid_base/distance/mean': (_C.LESS_EQUAL, 3.7),
    'complex/oxyanion_oah/distance/mean': (_C.LESS_EQUAL, 3.7),
    'complex/acid_base/dihedral2/mean': (_C.WITHIN_RANGE, (150.0, 210.0)),
    'complex/oxyanion_oah/dihedral2/mean': (_C.WITHIN_RANGE, (150.0, 210.0)),
}

SERINE_ESTERASE_BASE_THRESHOLDS = {
    'monomer/rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'monomer/protein_plddt_mean/seed_0': (_C.GREATER_EQUAL, 75.0),
    'monomer/motif_allatom_rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'monomer/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'monomer/predicted_distance_error/seed_0': (_C.LESS_EQUAL, 2.0),
    'complex/interface_lddt/seed_0': (_C.GREATER_EQUAL, 50.0),
    'complex/ligand_plddt_mean/seed_0': (_C.GREATER_EQUAL, 80.0),
    'complex/fixed_res_bb_aligned_allatom_sc_rmsd/seed_0': (_C.LESS_EQUAL, 3.0),
    'complex/motif_allatom_rmsd/seed_0': (_C.LESS_EQUAL, 1.5),
    'complex/chain_pair_pde_mean_sym/seed_0': (_C.LESS_EQUAL, 2.0),
    'complex/mean_pocket_bb_aligned_ligand_rmsd/seed_0': (_C.LESS_EQUAL, 5.0),
    'complex/mean_pocket_bb_rmsd/seed_0': (_C.LESS_EQUAL, 2.0),
    'sequence/seq_len': (_C.GREATER_EQUAL, 1),
    'sequence/max_single_aatype_frac': (_C.LESS_EQUAL, 0.25),
    'sequence/max_consecutive_identical_aatypes': (_C.LESS_EQUAL, 5),
    'sequence/num_blocklisted_motifs': (_C.LESS_EQUAL, 0),
    'sequence/unknown_count': (_C.LESS_EQUAL, 0),
    'sampled/topology_string_len': (_C.GREATER_EQUAL, 3),
    'sampled/num_missing_motif_residues': (_C.LESS_EQUAL, 0),
    'complex/num_missing_atoms/seed_0': (_C.LESS_EQUAL, 0),
}

SERINE_ESTERASE_HBOND_THRESHOLDS = {
    'monomer/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'monomer/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'es/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'es/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_fwd_oah1_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'complex/hbond_fwd_oah2_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_fwd_oah1_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'aei/hbond_fwd_oah2_oa/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_fwd_his_nuc/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_his_acid/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_fwd_oah1_ester/mean': (_C.GREATER_EQUAL, 0.01),
    'ti2/hbond_fwd_oah2_ester/mean': (_C.GREATER_EQUAL, 0.01),
}

SERINE_ESTERASE_THRESHOLDS = (
    SERINE_ESTERASE_BASE_THRESHOLDS
    | SERINE_ESTERASE_HBOND_THRESHOLDS
    | {
        'complex/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 90.4),
        'es_ti1_min_cross_state_rmsd': (_C.LESS_EQUAL, 1.0),
        'es_ti1_cross_seed_ligand_coord_std': (_C.LESS_EQUAL, 4.0),
    }
)

DEHP_ESTERASE_THRESHOLDS = (
    SERINE_ESTERASE_BASE_THRESHOLDS
    | SERINE_ESTERASE_HBOND_THRESHOLDS
    | {
        'complex/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 85.0),
        'aei/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 88.0),
        'ti2/ligand_plddt_mean/max': (_C.GREATER_EQUAL, 88.0),
    }
)

DEHP_ESTERASE_STRICT_THRESHOLDS = DEHP_ESTERASE_THRESHOLDS | {
    'complex/ligand_plddt_mean/mean': (_C.GREATER_EQUAL, 92.0),
    'aei/ligand_plddt_mean/mean': (_C.GREATER_EQUAL, 90.0),
    'ti2/ligand_plddt_mean/mean': (_C.GREATER_EQUAL, 92.0),
}

NITRENE_TRANSFERASE_THRESHOLDS = {
    'sampled/percent_ligand_bb_clashes': (_C.LESS_EQUAL, 5.0),
    'monomer/predicted_distance_error/seed_0': (_C.LESS_EQUAL, 2.0),
    'monomer/tm_score/seed_0': (_C.GREATER_EQUAL, 80.0),
    'heme_substrate/chain_pair_pde_mean_sym/seed_0': (_C.LESS_EQUAL, 2.0),
    'heme_substrate/interface_lddt/seed_0': (_C.GREATER_EQUAL, 30.0),
    'heme_substrate/mean_pocket_bb_aligned_ligand_rmsd/seed_0': (
        _C.LESS_EQUAL,
        2.5,
    ),
    'heme_substrate/mean_pocket_bb_rmsd/seed_0': (_C.LESS_EQUAL, 2.0),
    'sequence/max_consecutive_identical_aatypes': (_C.LESS_EQUAL, 5),
    'sequence/max_single_aatype_frac': (_C.LESS_EQUAL, 0.25),
    'sequence/num_blocklisted_motifs': (_C.LESS_EQUAL, 0),
    'sequence/unknown_count': (_C.LESS_EQUAL, 0),
    'heme_substrate/expanded_motif/motif_allatom_rmsd/seed_0': (
        _C.LESS_EQUAL,
        1.5,
    ),
    'heme_substrate/expanded_motif/motif_with_ligand_allatom_rmsd/seed_0': (
        _C.LESS_EQUAL,
        2.5,
    ),
    'sequence/seq_len': (_C.GREATER_EQUAL, 90),
    'heme_substrate/ligand_plddt_mean/seed_0': (_C.GREATER_EQUAL, 75.0),
    'heme_substrate/pocket_C9_min_atom_dist/seed_0': (
        _C.WITHIN_RANGE,
        (2.8, 3.8),
    ),
    'heme_substrate/pocket_C9_allatom_neighbors_mean/seed_0': (
        _C.WITHIN_RANGE,
        (38.0, 62.0),
    ),
    'heme_substrate/propionate_hbond_num_atoms/seed_0': (
        _C.WITHIN_RANGE,
        (3, 6),
    ),
    'heme_substrate/propionate_hbond_num_res/seed_0': (
        _C.WITHIN_RANGE,
        (3, 5),
    ),
    'heme_substrate/his_heme_dist/seed_0': (_C.LESS_EQUAL, 3.0),
    'heme_substrate/piperidine_propensity/min': (_C.LESS_EQUAL, 0.35),
    'n_self_rmsd/seed_0': (_C.LESS_EQUAL, 0.8),
}

## Kemp Eliminase

In [ ]:
KEMP_EVAL_DIR = '/tmp/kemp_campaign/04_eval'

kemp_df = load_eval_results(KEMP_EVAL_DIR)
kemp_df['passed'] = get_filter_mask(kemp_df, KEMP_ELIMINASE_THRESHOLDS)
kemp_summary = summarize_thresholds(kemp_df, KEMP_ELIMINASE_THRESHOLDS)
kemp_passing = kemp_df[kemp_df['passed']]

print(f'Passing Kemp eliminase designs: {len(kemp_passing)} / {len(kemp_df)}')
kemp_summary

In [ ]:
plot_pass_rates(kemp_summary)

sns.scatterplot(
    data=kemp_df,
    x='complex/tm_score/median',
    y='complex/mean_pocket_bb_aligned_ligand_rmsd/median',
    hue='passed',
)
plt.show()

## 4MU-Ac Serine Esterase

In [ ]:
SERINE_EVAL_DIR = '/tmp/serine_campaign/04_eval'

serine_df = load_eval_results(SERINE_EVAL_DIR)
serine_df['passed'] = get_filter_mask(serine_df, SERINE_ESTERASE_THRESHOLDS)
serine_summary = summarize_thresholds(serine_df, SERINE_ESTERASE_THRESHOLDS)
serine_passing = serine_df[serine_df['passed']]

print(
    f'Passing 4MU-Ac serine esterase designs: {len(serine_passing)} /'
    f' {len(serine_df)}'
)
serine_summary

In [ ]:
plot_pass_rates(serine_summary)

sns.scatterplot(
    data=serine_df,
    x='complex/ligand_plddt_mean/max',
    y='es_ti1_min_cross_state_rmsd',
    hue='passed',
)
plt.show()

## DEHP Esterase

In [ ]:
DEHP_EVAL_DIR = '/tmp/dehp_campaign/04_eval'

dehp_df = load_eval_results(DEHP_EVAL_DIR)
dehp_df['passed'] = get_filter_mask(dehp_df, DEHP_ESTERASE_THRESHOLDS)
dehp_df['passed_strict'] = get_filter_mask(
    dehp_df, DEHP_ESTERASE_STRICT_THRESHOLDS
)
dehp_summary = summarize_thresholds(dehp_df, DEHP_ESTERASE_THRESHOLDS)
dehp_passing = dehp_df[dehp_df['passed']]

print(
    f'Passing DEHP esterase designs: {len(dehp_passing)} / {len(dehp_df)}'
    f' (strict: {int(dehp_df["passed_strict"].sum())})'
)
dehp_summary

In [ ]:
plot_pass_rates(dehp_summary)

sns.scatterplot(
    data=dehp_df,
    x='complex/ligand_plddt_mean/max',
    y='aei/ligand_plddt_mean/max',
    hue='passed',
)
plt.show()

## Nitrene Transferase

In [ ]:
NITRENE_EVAL_DIR = '/tmp/nitrene_campaign/04_eval'

nitrene_df = load_eval_results(NITRENE_EVAL_DIR)
nitrene_df['passed'] = get_filter_mask(
    nitrene_df, NITRENE_TRANSFERASE_THRESHOLDS
)
nitrene_summary = summarize_thresholds(
    nitrene_df, NITRENE_TRANSFERASE_THRESHOLDS
)
nitrene_passing = nitrene_df[nitrene_df['passed']]

print(
    f'Passing nitrene transferase designs: {len(nitrene_passing)} /'
    f' {len(nitrene_df)}'
)
nitrene_summary

In [ ]:
plot_pass_rates(nitrene_summary)

sns.scatterplot(
    data=nitrene_df,
    x='heme_substrate/piperidine_propensity/min',
    y='n_self_rmsd/seed_0',
    hue='passed',
)
plt.show()